## Making the Final Dataset

Joining the separate csv's together then filter to only moisturizers then drop columns not needed for the analysis.

### Setup + Data Load

In [1]:
# Silently install kagglehub with pandas-datasets support
%pip install --q kagglehub[pandas-datasets]

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

In [3]:
DATASET = "nadyinky/sephora-products-and-skincare-reviews"

# Load products and look at the category options first
products = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, DATASET, "product_info.csv")
print(products.shape)

print(products[products["primary_category"] == "Skincare"]["secondary_category"].value_counts())

# Subcategories within Moisturizers, so I can decide which to keep
print(products[products["secondary_category"] == "Moisturizers"]["tertiary_category"].value_counts())

(8494, 27)
secondary_category
Moisturizers              551
Treatments                466
Cleansers                 361
Value & Gift Sets         196
Eye Care                  186
Masks                     166
Mini Size                 112
Sunscreen                 108
High Tech Tools            80
Wellness                   79
Lip Balms & Treatments     61
Self Tanners               53
Shop by Concern             1
Name: count, dtype: int64
tertiary_category
Moisturizers               386
Mists & Essences            67
Face Oils                   66
Night Creams                17
Decollete & Neck Creams     10
BB & CC Creams               4
Name: count, dtype: int64


**Secondary Category Decision:** 

Of the 551 products in the Moisturizers category, only the "Moisturizers" and "Night Creams" subcategories are kept. The others are excluded to keep comparisons like-for-like:
- Mists & Essences closer to toners, and judged differently by shoppers
- Face Oils a "greasy" complaint about an oil is almost expected, which would inflate greasy complaint rates
- Decollete & Neck Creams used on a different area of the body
- BB & CC Creams tinted products where complaints may be about shade or coverage rather than skin feel

### Filtering

In [4]:
# Subcategories (e.g., face oils) to exclude using the names from Cell 1's output
EXCLUDE_TERTIARY = ["Mists & Essences", "Face Oils", "Decollete & Neck Creams", "BB & CC Creams"]

# Columns to keep from the products and reviews datasets
product_cols = ["product_id", "product_name", "brand_name", "price_usd",
                "tertiary_category", "ingredients", "highlights"]

review_cols = ["product_id", "author_id", "rating", "submission_time",
               "skin_type", "review_title", "review_text"]

review_files = ["reviews_0-250.csv", "reviews_250-500.csv", "reviews_500-750.csv",
                "reviews_750-1250.csv", "reviews_1250-end.csv"]

# Filter products to moisturizers
is_moist = products["secondary_category"] == "Moisturizers"
not_excluded = ~products["tertiary_category"].isin(EXCLUDE_TERTIARY)
moist = products.loc[is_moist & not_excluded, product_cols]
moist_ids = set(moist["product_id"])
print(len(moist), "moisturizer products")

# Load each review file, keeping only moisturizer reviews
parts = []
for f in review_files:
    r = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS, DATASET, f,
        pandas_kwargs={"usecols": review_cols, "dtype": {"author_id": str, "product_id": str}},
    )
    parts.append(r[r["product_id"].isin(moist_ids)])
    print(f, "->", len(parts[-1]), "moisturizer reviews")

reviews = pd.concat(parts, ignore_index=True)
print("Total moisturizer reviews:", len(reviews))

# Create a unique review ID and remove duplicates
reviews["review_id"] = (reviews["author_id"].fillna("unknown") + "_"
                        + reviews["product_id"] + "_"
                        + reviews["submission_time"].astype(str))
print("Review IDs unique before dedup:", reviews["review_id"].is_unique)

before = len(reviews)
reviews = reviews.drop_duplicates(subset="review_id")
print("Removed", before - len(reviews), "duplicate reviews")

# Join product details onto reviews
df = reviews.merge(moist, on="product_id", how="left")
print("Final dataset:", df.shape)
print(df["skin_type"].value_counts(dropna=False))

404 moisturizer products
reviews_0-250.csv -> 128711 moisturizer reviews
reviews_250-500.csv -> 39080 moisturizer reviews
reviews_500-750.csv -> 19496 moisturizer reviews
reviews_750-1250.csv -> 20224 moisturizer reviews
reviews_1250-end.csv -> 7528 moisturizer reviews
Total moisturizer reviews: 215039
Review IDs unique before dedup: False
Removed 139 duplicate reviews
Final dataset: (214900, 14)
skin_type
combination    107622
dry             41557
normal          26227
oily            21938
NaN             17556
Name: count, dtype: int64


**Notes:** 

404 products remain after filtering (403 from the kept subcategories plus 1 product with a blank secondary category, removed later). Reviews are identified by a combination of reviewer, product, and submission time; 139 exact duplicates were removed. The skin type counts show combination skin dominates the data, which is why later analysis uses rates rather than counts.

In [5]:
# Eligibility Check: Count reviews per product and skin type, then filter to products with at least 30 reviews in every skin type
counts = (df.dropna(subset=["skin_type"])
            .groupby(["product_name", "brand_name", "skin_type"])
            .size()
            .unstack(fill_value=0))

MIN_PER_TYPE = 30
eligible = counts[(counts >= MIN_PER_TYPE).all(axis=1)]
print(len(eligible), "products have at least", MIN_PER_TYPE, "reviews in every skin type")
eligible.assign(total=eligible.sum(axis=1)).sort_values("total", ascending=False).head(20)

139 products have at least 30 reviews in every skin type


,skin_type,combination,dry,normal,oily,total
product_name,brand_name,,,,,
The True Cream Aqua Bomb,belif,3534,1075,801,886,6296
Protini Polypeptide Firming Refillable Moisturizer,Drunk Elephant,3377,1162,776,662,5977
Ultra Repair Cream Intense Hydration,First Aid Beauty,2704,1822,578,375,5479
The True Cream Moisturizing Bomb,belif,2115,1076,453,328,3972
The Water Cream Oil-Free Pore Minimizing Moisturizer,Tatcha,2244,469,424,762,3899
Magic Cream Moisturizer with Hyaluronic Acid,Charlotte Tilbury,1805,722,549,370,3446
Santorini Grape Poreless Skin Cream,KORRES,1857,511,460,322,3150
The Dewy Skin Cream Plumping & Hydrating Moisturizer,Tatcha,1435,1070,415,180,3100
Greek Yoghurt Nourishing Probiotic Gel-Cream,KORRES,1751,517,393,412,3073


**Duplicate Problem** 

The full-size and "Mini" versions of the same products (e.g., Estée Lauder Revitalizing Supreme+, Skinfix Barrier+) have identical review counts in every skin type. Sephora displays the same reviews on both product pages, so these reviews appear twice in the data under different product IDs. The next step removes them, keeping the full-size version.

### Cleaning

In [6]:
# From output above it's clear that Sephora uses the same reviews for mini and full-size products, so we want to keep only the full-size product's reviews.
print("Reviews before cross-product dedup:", len(df))
dupe_key = ["author_id", "submission_time", "review_text"]
print("Cross-product duplicates:", df.duplicated(subset=dupe_key).sum())

# Remove them, keeping the full-size product when a mini shares its reviews
df["is_mini"] = df["product_name"].str.contains("mini", case=False)
df = (df.sort_values("is_mini")                 # full-size rows first
        .drop_duplicates(subset=dupe_key, keep="first")
        .drop(columns="is_mini"))
print("Reviews after cross-product dedup:", len(df))

Reviews before cross-product dedup: 214900
Cross-product duplicates: 10298
Reviews after cross-product dedup: 204602


In [7]:
# Missing counts for each column
print(df.isna().sum())

author_id                0
rating                   0
submission_time          0
review_text            189
review_title         58117
skin_type            17133
product_id               0
review_id                0
product_name             0
brand_name               0
price_usd                0
tertiary_category       12
ingredients           4151
highlights           28039
dtype: int64


In [8]:
# Dropping reviews with no review text, since we want to analyze the text
df = df.dropna(subset=["review_text"])
print("Reviews after dropping empty review text:", len(df))

# Dropping reviews with missing skin type, since we want to analyze by skin type
print("Reviews missing skin type:", df["skin_type"].isna().sum()) 
df = df.dropna(subset=["skin_type"])
print("Reviews after dropping missing skin type:", len(df))       

# Dropping reviews with missing ingredients, since we want to analyze by ingredients
print("Reviews missing ingredients:", df["ingredients"].isna().sum())
df = df.dropna(subset=["ingredients"])
print("Reviews after dropping missing ingredients:", len(df))

# Dropping reviews with missing tertiary category, since we want to analyze by category
print("Reviews missing tertiary category:", df["tertiary_category"].isna().sum())
df = df.dropna(subset=["tertiary_category"])
print("Reviews after dropping missing tertiary category:", len(df))

Reviews after dropping empty review text: 204413
Reviews missing skin type: 17131
Reviews after dropping missing skin type: 187282
Reviews missing ingredients: 2606
Reviews after dropping missing ingredients: 184676
Reviews missing tertiary category: 12
Reviews after dropping missing tertiary category: 184664


**Cleaning Decisions:** 

Reviews are dropped when they are missing information the analysis depends on: review text (needed for text analysis), skin type (the core comparison), ingredients (product-level information), and subcategory (needed to confirm product type). Review titles and highlights are left as is, since a blank title is common and highlights are only a secondary check. Reviews without a skin type (about 8%) may differ from those who report it, which is a limitation of the analysis.

**Cleaning Summary Table**

| Step | Reviews remaining |
|---|---|
| Moisturizer reviews loaded, exact duplicates removed | 214,900 |
| Mini/full-size duplicates removed | 204,602 |
| Missing review text removed | 204,413 |
| Missing skin type removed | 187,282 |
| Missing ingredients removed | 184,676 |
| Missing subcategory removed | **184,664** |

In [9]:
# Eligibility Check (post cleaning): Count reviews per product and skin type, then filter to products with at least 30 reviews in every skin type
counts = (df.groupby(["product_name", "brand_name", "skin_type"])
            .size()
            .unstack(fill_value=0))

MIN_PER_TYPE = 30
eligible = counts[(counts >= MIN_PER_TYPE).all(axis=1)]
print(len(eligible), "products have at least", MIN_PER_TYPE, "reviews in every skin type")
eligible.assign(total=eligible.sum(axis=1)).sort_values("total", ascending=False).head(20)

130 products have at least 30 reviews in every skin type


,skin_type,combination,dry,normal,oily,total
product_name,brand_name,,,,,
The True Cream Aqua Bomb,belif,3525,1074,796,885,6280
Protini Polypeptide Firming Refillable Moisturizer,Drunk Elephant,3367,1159,774,659,5959
Ultra Repair Cream Intense Hydration,First Aid Beauty,2702,1822,578,375,5477
The True Cream Moisturizing Bomb,belif,2107,1072,451,324,3954
The Water Cream Oil-Free Pore Minimizing Moisturizer,Tatcha,2238,469,421,760,3888
Magic Cream Moisturizer with Hyaluronic Acid,Charlotte Tilbury,1804,722,549,370,3445
Santorini Grape Poreless Skin Cream,KORRES,1857,511,460,322,3150
The Dewy Skin Cream Plumping & Hydrating Moisturizer,Tatcha,1434,1068,414,179,3095
Greek Yoghurt Nourishing Probiotic Gel-Cream,KORRES,1751,517,393,412,3073


**Eligible Products:** 

After cleaning, 130 moisturizers have at least 30 reviews in every skin type, enough to compare skin types within the same product. The most-reviewed eligible products include several same brand pairs with different textures (e.g., belif Aqua Bomb vs. Moisturizing Bomb, Tatcha Water Cream vs. Dewy Skin Cream), which will be useful and interesting comparisons in Phase 3!

In [10]:
# Saving the final, cleaned dataset
out_file = "sephora_moisturizer_reviews_clean.csv.gz"
df.to_csv(out_file, index=False)
print(f"Saved {len(df)} reviews to {out_file}: {os.path.getsize(out_file) / 1e6:.1f} MB")

Saved 184664 reviews to sephora_moisturizer_reviews_clean.csv.gz: 31.6 MB
